In [ ]:
# model logging + json logging + semantic versioning + scaler.pkl (from table bytes)

import json
import pandas as pd
import cloudpickle
import os
import base64
import tempfile
import yaml

from snowflake.snowpark.functions import col
from snowflake.ml.model import custom_model
from snowflake.ml.registry import Registry

# ---------------------------
# Snowflake session
# ---------------------------
session = get_active_session()

stage_path = '@"ORANGE_ZONE_SBX_TA"."PUBLIC"."MY_CSV_STAGE"/feature_version_final.yml'

stream = session.file.get_stream(stage_path)
yaml_text = stream.read().decode("utf-8")

# Optional: parse YAML (for validation / versioning / logging)
yaml_dict = yaml.safe_load(yaml_text)

# ---------------------------
# Write YAML to temp file
# ---------------------------
yaml_local_path = os.path.join(
    tempfile.gettempdir(),
    "feature_version_final.yml"
)

with open(yaml_local_path, "w") as f:
    f.write(yaml_text)

print(f"YAML config loaded from stage and written to {yaml_local_path}")

# ---------------------------
# Custom Model Wrapper
# ---------------------------
class InterceptCustomModel(custom_model.CustomModel):
    def __init__(self, model):
        super().__init__(context=None)
        self.model = model

    @custom_model.inference_api
    def predict(self, input: pd.DataFrame) -> pd.DataFrame:
        output = self.model.predict(
            data=input,
            coeffs=self.model.model_coefficients_final
        )
        return pd.DataFrame({"prediction": output})

# ---------------------------
# Initialize Registry
# ---------------------------
reg = Registry(
    session=session,
    database_name="ORANGE_ZONE_SBX_TA",
    schema_name="REGISTRY"
)

# ---------------------------
# Fetch intercept rows (MODEL_BYTES + SCALER_BYTES)
# ---------------------------
intercept_rows = (
    session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS")
    .filter(col("PARAMETERS") == "Intercept")
    .select(
        "REGIONNAME",
        "F_CODE",
        "MODEL_BYTES",
        "SCALER_BYTES",
        "WMAPE",
        "MAPE",
        "RMSE",
        "BIAS",
        "TRACKING_SIGNAL",
        "ALPHA",
        "LAMBDA",
        "TRAIN_START_DATE",
        "TRAIN_END_DATE",
        "LOAD_TS",
    )
    .collect()
)

# ---------------------------
# Loop over models
# ---------------------------
for row in intercept_rows:

    region = row["REGIONNAME"]
    f_code = row["F_CODE"]
    region_fcode = f"{region}_{f_code}"
    region_fcode_safe = region_fcode.replace("-", "_")

    print(f"\nProcessing model: {region_fcode}")

    # ---------------------------
    # Decode MODEL from bytes
    # ---------------------------
    model_bytes = base64.b64decode(row["MODEL_BYTES"])
    model_object = cloudpickle.loads(model_bytes)

    # ---------------------------
    # Decode SCALER from bytes
    # ---------------------------
    scaler_artifact_path = None

    if row["SCALER_BYTES"] is not None:
        scaler_bytes = base64.b64decode(row["SCALER_BYTES"])
        scaler_object = cloudpickle.loads(scaler_bytes)

        scaler_artifact_path = os.path.join(
            tempfile.gettempdir(),
            f"scaler_{region_fcode_safe}.pkl"
        )

        with open(scaler_artifact_path, "wb") as f:
            cloudpickle.dump(scaler_object, f)

        print(f"Scaler artifact written: {scaler_artifact_path}")
    else:
        print("SCALER_BYTES is NULL — skipping scaler artifact")

    # ---------------------------
    # Wrap model
    # ---------------------------
    custom_model_instance = InterceptCustomModel(model_object)

    # ---------------------------
    # Fetch feature list
    # ---------------------------
    features_rows = (
        session.table("PUBLIC.PROD_FINAL_MODEL_LASSO_COEFFICIENTS")
        .filter(
            (col("REGIONNAME") == region) &
            (col("F_CODE") == f_code) &
            (col("PARAMETERS") != "Intercept")
        )
        .select("PARAMETERS")
        .collect()
    )

    features = sorted({r["PARAMETERS"] for r in features_rows})

    # ---------------------------
    # Semantic versioning
    # ---------------------------
    try:
        with open(f"model_metadata_{region_fcode}.json", "r") as f:
            last_metadata = json.load(f)

        last_features = last_metadata["features"]
        last_train_period = last_metadata["train_period"]
        last_version = last_metadata.get("version", "v_0_0")

        major, minor = map(int, last_version.replace("v_", "").split("_"))

        if features != last_features:
            major += 1
            minor = 0
        elif (
            str(row["TRAIN_START_DATE"]) != last_train_period["start"]
            or str(row["TRAIN_END_DATE"]) != last_train_period["end"]
        ):
            minor += 1

        version = f"v_{major}_{minor}"

    except FileNotFoundError:
        version = "v_0_0"

    # ---------------------------
    # Model metadata
    # ---------------------------
    model_metadata = {
        "name": region_fcode,
        "model_type": "ElasticNet",
        "description": f"ElasticNet demand model for {region_fcode}",
        "version": version,
        "metrics": {
            "wmape": float(row["WMAPE"])*100,
            "mape": float(row["MAPE"]),
            "rmse": float(row["RMSE"]),
            "bias": float(row["BIAS"]),
            "tracking_signal": float(row["TRACKING_SIGNAL"]),
        },
        "hyperparameters": {
            "alpha": float(row["ALPHA"]),
            "lambda": float(row["LAMBDA"]),
        },
        "train_period": {
            "start": str(row["TRAIN_START_DATE"]),
            "end": str(row["TRAIN_END_DATE"]),
        },
        "features": features,
        "training_date": str(row["LOAD_TS"]),
        "tags": {
            "region": region,
            "f_code": f_code,
            "algorithm": "ElasticNet",
        },
        "owner": "tiger_analytics_team",
        "training_wh": "TA_USER_BASE_WH",
        "artifact_stage": "@PROD_TABLES.SCRIPTS",
    }

    # ---------------------------
    # Write metadata JSON
    # ---------------------------
    metadata_path = os.path.join(
        tempfile.gettempdir(),
        f"model_metadata_{region_fcode_safe}.json"
    )

    with open(metadata_path, "w") as f:
        json.dump(model_metadata, f, indent=2)

    # ---------------------------
    # Log model to registry
    # ---------------------------
    model_name = f"{region_fcode_safe}"

    mv = reg.log_model(
        model=custom_model_instance,
        model_name=model_name,
        conda_dependencies=["scikit-learn", "pandas"],
        options={"relax_version": False},
        user_files={
            "metadata": [metadata_path],
            "scaler": [scaler_artifact_path] if scaler_artifact_path else [],
            "config": [yaml_local_path],
        },
        comment=f"Elastic Model for {region_fcode}",
        sample_input_data=pd.DataFrame(model_object.X),
    )

    # ---------------------------
    # Alias management
    # ---------------------------
    model_ref = reg.get_model(model_name)
    latest_version_name = mv.version_name

    for v in model_ref.versions():
        if v.version_name == latest_version_name:
            continue

        try:
            v.unset_alias("PROD")
        except Exception:
            pass

        try:
            v.unset_alias("ARCHIVED")
        except Exception:
            pass

        v.set_alias("ARCHIVED")

    mv.set_alias("PROD")

    print(
        f"Model {model_name} | Registry version {mv.version_name} "
        f"| Semantic version {version} promoted to PROD"
    )

    print(f"Successfully logged {region_fcode}")
